# Diamond-Square (heightmaps)

**Domain:** Procedural Generation  ·  *recommended addition*  ·  **runnable:** yes

A compact refresher on the classic fractal-terrain algorithm: how it works, why it
looks the way it does, where it bites, and what to use instead.

## 1. What & Why

**Diamond-Square** is a fast, deterministic algorithm for generating natural-looking
**heightmaps** — 2D grids of elevation values you can render as terrain. It is the 2D
generalization of *random midpoint displacement* (the 1D fractal-coastline trick) and is
sometimes called the **plasma fractal** or **cloud** algorithm.

**The problem it solves.** You want terrain that looks plausibly natural — ridges,
valleys, rolling hills — without hand-authoring every pixel. Pure white noise looks like
TV static; a smooth function looks fake. Real terrain is *fractal*: it has detail at every
scale, and zooming in reveals structure similar to zooming out (statistical
self-similarity). Diamond-Square builds exactly that by recursively subdividing a grid and
adding **random displacement whose magnitude shrinks with each level of detail**.

**When to reach for it.**

- You need a quick, cheap heightmap and have a power-of-two-plus-one grid (e.g. 129×129).
- Prototyping terrain for a game, a visualization, or a roguelike world map.
- Teaching/illustrating fractal Brownian motion — it is the simplest thing that works.

**When *not* to.** If you need tileable/infinite terrain, fine artistic control, or
artifact-free output, reach for **Perlin/Simplex/value noise** with fractal Brownian
motion (fBm) instead. Diamond-Square is locked to a fixed square grid and has a known
axis-aligned creasing artifact (see Gotchas).

## 2. Mental Model

Picture a square rubber sheet pinned at its **four corners** at random heights. Now
repeatedly do two things, halving the "jitter budget" each round:

```
DIAMOND step           SQUARE step
C . . . C              C . E . C       C = corner / known point
. . . . .              . . . . .       M = square midpoint (diamond step)
. . M . .   ----->     E . M . E       E = edge midpoint   (square step)
. . . . .              . . . . .
C . . . C              C . E . C
```

1. **Diamond step** — the *center* of each square is set to the average of its 4 **corners**
   plus a random nudge. (The sampled points form diamond shapes.)
2. **Square step** — each *edge midpoint* is set to the average of the 4 **diamond points
   around it** plus a random nudge. (The sampled points form squares.)

After both steps the grid is twice as dense. Repeat on the new, smaller squares. Crucially,
the random nudge **gets smaller every iteration** — big moves carve the mountains and
valleys early, tiny moves add fine roughness late. That decaying-displacement loop is the
whole trick, and it is what makes the result fractal.

## 3. Key Concepts

- **Grid size must be 2ⁿ + 1.** The algorithm always subdivides the *interval between*
  points, so you need an odd side length: 3, 5, 9, 17, 33, 65, 129, 257, …. A 128×128 grid
  will not line up.
- **Two alternating passes.** *Diamond* fills square centers from corners; *square* fills
  edge midpoints from diamond centers. They must alternate — square depends on diamond.
- **Step size.** Each iteration the spacing between active points halves (`step //= 2`).
  The loop ends when `step == 1` and every cell is filled.
- **Roughness / Hurst exponent (H).** The random amplitude is multiplied by `2^(-H)` (often
  written `0.5 ** roughness`) each iteration. **High roughness → amplitude decays fast →
  smooth, rolling terrain. Low roughness → slow decay → jagged, spiky terrain.** This is the
  single most important knob.
- **Edge wrapping.** Square-step points on the grid border are missing a neighbor. You
  either average the 3 available neighbors or wrap around to the opposite edge (wrapping
  yields a tileable map).
- **Self-similarity / fBm.** The output approximates **fractional Brownian motion** with a
  fractal dimension controlled by H — the mathematical reason it reads as "natural".

## 4. Setup

Pure-Python + NumPy; no special libraries. NumPy does the array bookkeeping and
Matplotlib renders the heightmap. Both are CPU-only and tiny here.

In [ ]:
# %pip install numpy matplotlib
import numpy as np

rng = np.random.default_rng(42)  # fixed seed -> reproducible terrain
print("numpy", np.__version__)

## 5. Worked Examples

### Example 1 — the algorithm, end to end

A direct, readable implementation. We work in-place on a `(2ⁿ+1)`² float array, seeding the
four corners and then running the diamond/square loop while halving both the step and the
random amplitude.

In [ ]:
def diamond_square(n, roughness=1.0, rng=rng):
    """Generate a (2**n + 1) square heightmap via diamond-square.

    roughness: higher -> amplitude decays faster -> smoother terrain.
    """
    size = 2 ** n + 1
    h = np.zeros((size, size), dtype=float)

    # Seed the four corners with random heights.
    for y in (0, size - 1):
        for x in (0, size - 1):
            h[y, x] = rng.uniform(-1, 1)

    step = size - 1
    amp = 1.0
    while step > 1:
        half = step // 2

        # --- Diamond step: square centers from their 4 corners ---
        for y in range(half, size, step):
            for x in range(half, size, step):
                avg = (h[y - half, x - half] + h[y - half, x + half] +
                       h[y + half, x - half] + h[y + half, x + half]) / 4.0
                h[y, x] = avg + rng.uniform(-amp, amp)

        # --- Square step: edge midpoints from their (up to) 4 diamond neighbors ---
        for y in range(0, size, half):
            # offset x so we hit the points the diamond step left empty
            for x in range((y + half) % step, size, step):
                total, count = 0.0, 0
                for dy, dx in ((-half, 0), (half, 0), (0, -half), (0, half)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < size and 0 <= nx < size:
                        total += h[ny, nx]
                        count += 1
                h[y, x] = total / count + rng.uniform(-amp, amp)

        step = half
        amp *= 0.5 ** roughness  # shrink displacement for the next, finer level

    return h


hmap = diamond_square(n=7, roughness=0.9)  # 129 x 129
print("shape:", hmap.shape)
print(f"height range: {hmap.min():.3f} .. {hmap.max():.3f}")

The grid is `2**7 + 1 = 129` on a side and every cell is filled. Now render it as a shaded
heightmap so the fractal structure is visible.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; no display server needed
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(hmap, cmap="terrain")
ax.set_title("Diamond-Square heightmap (n=7, roughness=0.9)")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="elevation")
fig.tight_layout()
fig.savefig("diamond_square_heightmap.png", dpi=90)
print("saved diamond_square_heightmap.png")

### Example 2 — roughness controls the terrain character

The roughness exponent is the knob you actually tune. Generate the same grid at three
values and compare a simple "ruggedness" metric — the mean absolute slope between adjacent
cells. Lower roughness keeps more high-frequency displacement, so the surface is rougher.

In [ ]:
def ruggedness(h):
    """Mean absolute difference between neighboring cells (higher = more jagged)."""
    dx = np.abs(np.diff(h, axis=1)).mean()
    dy = np.abs(np.diff(h, axis=0)).mean()
    return (dx + dy) / 2

maps = {}
for r in (0.4, 0.9, 1.5):
    m = diamond_square(n=7, roughness=r, rng=np.random.default_rng(42))
    maps[r] = m
    print(f"roughness={r:<4}  ruggedness={ruggedness(m):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (r, m) in zip(axes, maps.items()):
    ax.imshow(m, cmap="terrain")
    ax.set_title(f"roughness = {r}")
    ax.axis("off")
fig.tight_layout()
fig.savefig("diamond_square_roughness.png", dpi=90)
print("\nlow roughness -> jagged/spiky; high roughness -> smooth/rolling")

As expected, the ruggedness metric falls as roughness rises: at `roughness=0.4` the
amplitude barely decays so fine detail dominates; at `roughness=1.5` the displacement
collapses quickly and you get smooth, rounded hills.

## 6. Gotchas & Pitfalls

- **Wrong grid size.** It *must* be `2ⁿ + 1`. Using a power-of-two side (256, 512) leaves
  the center/edge indices off-by-one and the recursion never aligns. Always size as
  `2**n + 1`.
- **Diamond/square order.** The square step reads the points the diamond step just wrote.
  Swap them, or skip the alternation, and you get garbage or holes. Note the `(y + half) %
  step` offset in the square loop — that staggering is what hits exactly the still-empty
  cells.
- **Forgetting to decay the amplitude.** If `amp` does not shrink each iteration the result
  is just white noise — no fractal structure. The decay (`amp *= 0.5 ** roughness`) is the
  algorithm.
- **The square-grid creasing artifact.** Diamond-Square is famous for faint, regular
  ridges/seams aligned to the grid axes and along the boundaries of the largest squares.
  They come from the fixed subdivision pattern. Mitigations: add extra jitter at coarse
  levels, use the *diamond-square-with-seeded-randomness* variants, or post-smooth — but if
  artifacts matter, Perlin/Simplex noise avoids them entirely.
- **Edge handling changes the look.** Averaging 3 neighbors at the border gives a free,
  non-tileable edge; wrapping to the opposite side gives a seamless/tileable map but ties
  the four corners together. Pick deliberately.
- **O(N²) and pure-Python loops are slow.** A 1025×1025 map is ~1M cells; the nested-loop
  version above is fine for ≤257 but vectorize (slice assignments) or drop to a compiled
  language for large maps.
- **Unbounded output range.** Heights are not normalized to [0, 1]. Rescale
  (`(h - h.min()) / (h.max() - h.min())`) before mapping to colors, sea level, or tile
  thresholds.

## 7. When to Use vs Alternatives

| Approach | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|
| **Diamond-Square** | Dead simple, fast, fractal out of the box | Fixed `2ⁿ+1` grid, grid-axis creasing, not naturally tileable | You need a quick standalone heightmap and don't mind a square grid |
| **Perlin / Simplex noise + fBm** | Artifact-free, smooth gradients, sample at any (x, y), tileable, infinite | More code/intuition; needs octave summing for fractal look | Production terrain, infinite/streaming worlds, fine control |
| **Value / midpoint noise** | Conceptually close to Diamond-Square, easy | Blockier than gradient noise | Teaching, very cheap textures |
| **Worley (cellular) noise** | Great for cells, cracks, biomes, Voronoi-like features | Not a smooth heightmap on its own | Caves, stone, region masks |
| **Real DEM data** | Actually real terrain | Not procedural; storage; licensing | You need true-to-life geography |

**Rule of thumb:** prototype with Diamond-Square because it is the fastest thing to get a
credible heightmap on screen; graduate to **Perlin/Simplex + fBm** the moment you need
tiling, arbitrary sampling, or want the grid-aligned creases gone. They share the same fBm
DNA — roughness/octaves/persistence are the same idea under different names. See the
Voronoi/Delaunay notebook for cellular partitioning and the Poisson-disk notebook for
blue-noise point scattering.

## 8. Resources

- **Original paper** — Fournier, Fussell & Carpenter, "Computer Rendering of Stochastic
  Models" (CACM 1982), which introduced midpoint-displacement terrain:
  <https://dl.acm.org/doi/10.1145/358523.358553>
- **The Diamond-Square algorithm** (clear visual walk-through), Paul Martz / GameDev:
  <https://web.archive.org/web/20120907112705/http://gameprogrammer.com/fractal.html>
- **Wikipedia — Diamond-square algorithm** (pseudocode + artifact discussion):
  <https://en.wikipedia.org/wiki/Diamond-square_algorithm>
- **"The Perlin noise math FAQ"** and Red Blob Games' **Making maps with noise functions**
  for the gradient-noise alternative and fBm intuition:
  <https://www.redblobgames.com/maps/terrain-from-noise/>
- **Fractal Brownian Motion** background (Inigo Quilez):
  <https://iquilezles.org/articles/fbm/>